### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="musk_iid",
    version_from_unique_name="musk",
    version_comment="IID outer splits for the musk dataset",
    # -- Rest as before
    dataset_year="1994",
    domain_str="chemistry & material science",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C51608",
    download_description="""
wget https://archive.ics.uci.edu/static/public/75/musk+version+2.zip && unzip musk+version+2.zip clean2.data.Z && uncompress clean2.data.Z && rm musk+version+2.zip && mkdir -p local-data-warehouse/musk && mv clean2.data local-data-warehouse/musk/
""",
    # References
    academic_reference_bibtex="""@article{dietterich1993comparison,
  title={A comparison of dynamic reposing and tangent distance for drug activity prediction},
  author={Dietterich, Thomas and Jain, Ajay and Lathrop, Richard and Lozano-Perez, Tomas},
  journal={Advances in neural information processing systems},
  volume={6},
  year={1993}
}
""",
    academic_reference_bibtex_key="dietterich1993comparison",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
- We rename the molecule IDs to remove the target leakage from the names.
- We drop the conformation name as it leaks information that the real task should not have (the correlation between specific conformations across samples).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="class",
)

## Preprocessing

In [2]:
import pandas as pd
import uuid

df = pd.read_csv(dataset_mold.path / "clean2.data", header=None, names=[
    "molecule_name", "conformation_name", *[f"feature_{i}" for i in range(166)], "class",
])
print("Loaded data shape:", df.shape)

df = df.drop(columns=["conformation_name"])

df["class"] = df["class"].map({0: "non-musk", 1: "musk"})

# Create mapping: molecule -> random string id
mapping = {val: uuid.uuid4().hex[:12] for val in df["molecule_name"].unique()}
df["molecule_name"] = df["molecule_name"].map(mapping)

as_cat_type = ["molecule_name", "class"]
df[as_cat_type] = df[as_cat_type].astype("category")


df = df.sample(frac=1, random_state=42).sort_values(by="molecule_name").reset_index(drop=True)

Loaded data shape: (6598, 169)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 6,598
Columns: 168
Use sampling: False (sample size: 6,598)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['feature_126', 'feature_72', 'feature_33', 'feature_20', 'feature_132', 'feature_55', 'feature_128', 'feature_139', 'feature_59', 'feature_47']
Rows remaining as candidates after top-10 filter: 860 (of 6,598)

#### Duplicate Report
Total duplicate rows: 17 (0.26% of dataset)
Duplicate rows ignoring target: 17 (0.26% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,molecule_name,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,feature_80,feature_81,feature_82,feature_83,feature_84,feature_85,feature_86,feature_87,feature_88,feature_89,feature_90,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100,feature_101,feature_102,feature_103,feature_104,feature_105,feature_106,feature_107,feature_108,feature_109,feature_110,feature_111,feature_112,feature_113,feature_114,feature_115,feature_116,feature_117,feature_118,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128,feature_129,feature_130,feature_131,feature_132,feature_133,feature_134,feature_135,feature_136,feature_137,feature_138,feature_139,feature_140,feature_141,feature_142,feature_143,feature_144,feature_145,feature_146,feature_147,feature_148,feature_149,feature_150,feature_151,feature_152,feature_153,feature_154,feature_155,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,class
0,00b64ae2feb3,2,-198,-159,146,-117,-122,52,37,-25,-129,-49,-124,-58,-97,-76,-293,37,-72,-69,-80,-90,-12,-66,-65,-94,6,-10,-91,65,-141,-116,-18,77,-233,83,47,-173,-10,-145,-29,-183,-138,-80,-57,-128,-120,-115,-74,-23,-127,-107,-16,-14,52,134,-70,62,-151,-12,-169,111,-187,2,-185,-12,26,-166,-116,-132,-113,-168,-150,-50,-133,-61,-179,8,-54,-103,55,-82,14,-59,207,-181,40,-137,-74,119,-145,-201,18,68,-135,46,-33,-41,37,-155,42,-205,-5,-53,-50,-85,-131,-170,-92,-69,-124,-48,-73,-68,-51,-196,-113,-78,57,72,-20,-111,50,-103,107,16,5,-157,-189,-132,-48,58,-116,-72,-119,-99,-109,39,-106,-48,-41,-36,-40,22,14,-178,-104,-120,-117,-129,-11,126,-2,67,-170,-121,-193,-236,-265,-206,-130,6,9,148,-50,-112,83,musk
1,00b64ae2feb3,22,-197,-151,148,-117,-50,66,-6,-20,-174,-50,-101,-88,-85,-95,-279,34,-100,-94,-41,-51,-10,-81,-60,-93,-17,18,-120,6,-144,-116,-4,53,-237,91,50,-174,-39,-145,18,-180,-104,-121,-52,-123,-107,-107,-63,-42,-136,-68,-24,-19,37,22,-60,84,-152,2,-185,135,-180,41,-185,-26,13,-166,-114,-133,-82,-163,-150,-68,-111,-97,-180,8,-27,-105,66,-57,31,-59,205,-193,38,-125,-134,79,-143,-202,10,87,-152,41,91,-71,9,-155,73,-203,20,-79,-61,-90,-63,-127,-52,-74,-124,-40,-58,-65,-52,-178,-72,-88,35,78,-9,-111,-74,-95,40,-59,21,-181,-195,-165,-11,53,108,-74,-96,-75,-133,3,-122,-71,-48,-74,-13,51,8,-178,-104,-120,-92,-111,28,124,-11,64,-169,-20,-186,-236,-258,-206,-134,-90,-15,138,-58,-119,39,musk
2,00b64ae2feb3,20,-198,-159,32,-117,131,52,40,-24,-127,-51,-125,-58,-98,-74,-296,37,-73,-70,-81,-90,-12,-66,-62,-94,6,-8,-90,115,-141,-116,-6,69,-211,70,53,43,-12,-145,-27,-183,-99,-79,-59,-126,-126,-115,-75,-22,-124,-107,-16,-15,56,-21,-71,63,-151,137,-170,-42,-181,39,-97,87,11,-166,-117,-132,-112,-168,-150,-51,-133,-59,-180,8,-58,-104,55,-82,14,-60,-39,-181,40,-136,-73,71,12,-202,8,49,-139,42,92,118,36,-154,43,-205,21,-52,-50,-85,-137,-170,-92,-67,-124,-51,-74,-68,-51,-195,-113,-89,57,73,-19,-113,73,78,46,-78,29,-158,-161,-134,30,56,108,-72,-118,-100,-110,38,-106,-48,-39,-36,-40,22,14,-178,-104,-119,-116,-128,-9,126,60,58,-169,-18,-185,-238,-270,-201,-133,-97,-14,141,-60,-126,21,musk

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,molecule_name,category,0.0,0.0,102.0,"c394ef9107ce, a470dec69277, d2c2629403c4, 6c75e31d113c, a24523edbb05, 14b0c39debce, 54c36f77003d, 6047a5025b9c, 311bb792dd49, 0208e864f467"
1,class,category,0.0,0.0,2.0,"non-musk, musk"
2,feature_0,int64,0.0,0.0,202.0,"44, 43, 35, 36, 46, 51, 48, 37, 47, 57"
3,feature_1,int64,0.0,0.0,260.0,"-198, -194, -199, -193, -192, -195, -196, -197, 86, -191"
4,feature_2,int64,0.0,0.0,221.0,"-145, -144, -112, -19, -22, -111, 31, -62, -23, -146"
5,feature_3,int64,0.0,0.0,257.0,"-76, -77, -69, 28, -70, 29, 33, 131, 32, 152"
6,feature_4,int64,0.0,0.0,129.0,"-117, -116, -115, -113, -112, -111, -114, -108, -110, -109"
7,feature_5,int64,0.0,0.0,358.0,"11, 12, 10, 86, 85, 54, -154, 55, 53, 52"
8,feature_6,int64,0.0,0.0,323.0,"56, 26, 57, -163, 27, -160, -162, -161, -164, -159"
9,feature_7,int64,0.0,0.0,389.0,"-95, -96, -103, 57, 64, -3, -171, -102, -5, 67"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
feature_0,6598.0,58.945135,53.249007,-31.0,292.0
feature_1,6598.0,-119.128524,90.813375,-199.0,95.0
feature_2,6598.0,-73.146560,67.956235,-167.0,81.0
feature_3,6598.0,-0.628372,80.444617,-114.0,161.0
feature_4,6598.0,-103.533495,64.387559,-118.0,325.0
feature_5,6598.0,18.359806,80.593655,-183.0,200.0
feature_6,6598.0,-14.108821,115.315673,-171.0,220.0
feature_7,6598.0,-1.858290,90.372537,-225.0,320.0
feature_8,6598.0,-86.003031,108.326676,-245.0,147.0
feature_9,6598.0,-44.495756,72.088903,-286.0,231.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column        rank                            
class         1         non-musk   5581  84.59
              2             musk   1017  15.41
molecule_name 1     c394ef9107ce   1044  15.82
              2     a470dec69277   1010  15.31
              3     d2c2629403c4    911  13.81
              4     6c75e31d113c    383   5.80
              5     a24523edbb05    344   5.21

In [8]:
# Target Distribution
target_df

,count,pct
class,,
non-musk,5581,84.59
musk,1017,15.41


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default IID splits",
    splits=splits
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to musk/versions/019dd49e-4009-7822-a3a5-bb615a182e0f
019dd49e-4009-7822-a3a5-bb615a182e0f
7bdff1ae335fde55bc681ed16627ff0e0fa2471272e9b6636943c2e4d2727906
